## Tutorial: Elastic inverse problems

This tutorial uses `triangulax` to study a geometric inverse problem: find a dynamical process that generates a target shape $\mathcal{S}_1$ from an initial shape $\mathcal{S}_1$. In this setting, a simulation is effectively treated like a "neural network" which maps initial conditions to simulation results, and simulation parameters as trainable "weights". We saw a very simple inverse problem in tutorial 3: now we will consider a more complex problem in 3d.

For example, consider an elastic energy $E (\mathbf{v};  \boldsymbol{\theta})$ that depends on both mesh vertex positions $\mathbf{v};$ and parameters ${\theta})$ (like elastic moduli or reference shapes). 
If vertices move to relax the elastic energy, the final state is the minimum-energy configuration $\mathbf{v}^*(\boldsymbol{\theta}) = \arg \min E ( \cdot ;\theta)$. The goal is to find parameters $\boldsymbol{\theta}^*$ so that the equilibrated vertex positions $\mathbf{v}^*(\boldsymbol{\theta}^*)$ match the target shape. Via automatic differentiation, one can differentiate $\mathbf{v}^*(\boldsymbol{\theta})$ with respect to the parameters $\boldsymbol{\theta}$, and use gradient-based optimizers like ADAM to find the desired parameter value $\boldsymbol{\theta}^*$.

The optimization library `optimistix` and the ODE library `diffrax` allow automatically differentiating through complex simulations. More generally, differentiating through a simulation facilitates sensitivity analyses and fitting simulations to (experimental) data.

### Growing Suzanne

As a toy problem, let's build a simple biophysical model that produces "Suzanne", a mesh model of a monkey's head that is a common test case in the 3d software Blender:

<div>
<img src="suzanne.png" width="500"/>
</div>

Inspired by models for the growing leafs and tissue sheets, we will treat the mesh as a two-layered elastic material. 
The reference/target shape of the elastic sheet evolves in time due to growth, and the surface deforms to minimize its elastic energy. Differences in the growth rates of the two layers (inner and outer) induce curvature.

#### Elastic energy

The elastic energy combines an in-plane and a bending contribution. For the former, we use the neo-hookean energy from tutorial 5:

$$E_{\mathrm{NH}} = \delta \sum_f A_f \left[ \frac{K}{2}\left(\frac{\mathrm{tr}(C_f)}{\sqrt{\det C_f}} - 2\right)
+\frac{G}{2}\left(\sqrt{\det C_f} - 1\right)^2
\right], \qquad C_f = G_0^{-1} G$$

Here, $C_f$ is the Cauchy-Green tensor (a nonlinear measure of strain), computed from $G$, $G_0$ the current and reference metric tensors for each triangle $f$. $G_0$ defines the target shape of each triangle.
The total energy sums over all faces, weighted by area $A_f$. $K$ and $G$ are the bulk and shear moduli. The overall scale is set by the shell thickness $\delta$.

Bending resistance is modeled using the dihedral angles $\theta_{e}$ for each edge:
$$E_{\mathrm{B}} = \delta^3 \sum_e \frac{\ell_e^2}{A_e} (\theta_e -\theta_e^0)^2  $$
Where the area $A_e$ is defined as $1/3$ of the areas of the adjacent triangles. For thin shells, bending rigidity is thus much weaker than resistance to in-plane deformation (as one may verify using a sheet of paper).

To model growth, we allow the reference metric $G_f^0(t)$ to depend on time. By convention, $t=0$ is the initial and $t=1$ the final timepoint. For simplicity, we focus on isotropic growth with rate $\lambda_f$: 
$$ G_f^0(t) = e^{\lambda_f t} G_f^0(0) $$
At each timestep, we compute the new elastic energy minimum shape from the preceding one.
Differences in growth rate between the two layers lead to bending, which


The goal is find $\lambda_f$ so that the elastic energy equilibrium at the final time matches the target shape.

#### Loss function

Now, we need a loss function that measures the agreement between the simulation output and the target. 
This can be a non-trivial problem, because we need to measure the similarity of meshes with different number of vertices and connectivity.

We first align the output and target meshes


In [ ]:
# On second thought, the isotropic growth problem can be "analytically" solved by finding a conformal map from
# S_0 and S_1 to the unit sphere, and using the relative (area) strain as the growth factor. 
# Otoh, the bending energy constraint may make this non-trivial.

# Other ideas: use nematic (anisotropic growth)
# solve reverse dynamical problem (starting from target shape)

In [7]:
import numpy as np
from scipy import sparse, optimize, ndimage
import matplotlib.pyplot as plt
import meshplot

import igl

In [2]:
import jax.numpy as jnp
import jax

In [4]:
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_debug_nans", True)

In [3]:
from jaxtyping import Float

In [ ]:
import equinox

In [5]:
import lineax
import optimistix
import optax

In [6]:
from triangulax import trigonometry as trig
from triangulax import geometry as geom
from triangulax import adjacency as adj
from triangulax import linops as lin
from triangulax.triangular import TriMesh
from triangulax.mesh import HeMesh, GeomMesh
from triangulax import linops
from triangulax import algorithms as algo

### Load mesh

In [8]:
# let's use a fine mesh of a sphere as out initial condition and reference configuration
# tutorial_meshes/sphere_finer.obj

### Define mesh-based energies

In [ ]:
# --- In-plane elastic energy ---

def get_metric(vertices: Float[jax.Array, "n_faces dim"],
               hemesh: HeMesh) -> Float[jax.Array, "n_faces 2 2"]:
    """Metric tensor (first fundamental form) per triangle."""
    a, b, c = vertices[hemesh.faces.T]
    J = jnp.stack([b - a, c - a], axis=1)
    return jnp.einsum("vik,vjk->vij", J, J)


def get_neo_hookean_energy_density(C: Float[jax.Array, "2 2"], mod_bulk: float, mod_shear: float
                                   ) -> Float[jax.Array, ""]:
    """Compute the neo-Hookean energy density from Cauchy-Green deformation tensor C:
        E = mod_shear/2 * (trace(C) / J - 2) + mod_bulk/2 * (J - 1)^2
    where J = sqrt(det(C)) is the local area change.
    See https://en.wikipedia.org/wiki/Neo-Hookean_solid
    """
    J = jnp.sqrt(jnp.linalg.det(C))
    return mod_shear/2*(jnp.trace(C)/J - 2) + mod_bulk/2*(J-1)**2


@jax.jit
def get_neo_hookean_energy(vertices, args):
    """Total neo-Hookean energy. args = (hemesh, metric_ref, mod_bulk, mod_shear)"""
    hemesh, metric_ref, mod_bulk, mod_shear = args
    areas = geom.get_triangle_areas(vertices, hemesh)
    C = jnp.einsum("fij,fjk->fik", jnp.linalg.inv(metric_ref), get_metric(vertices, hemesh))
    return (areas * jax.vmap(lambda c: get_neo_hookean_energy_density(c, mod_bulk, mod_shear))(C)).sum()


# --- Bending energy ---


# to do: bending energy using dihedral angles: \sum_ij (\theta_{ij}-\theta_{ij}^0)^2 * l_ij^2 / a_ij
# where a_ij is the average of the adjacent triangle areas / 3

### Load target mesh and compute distance transform